# Net Productivity Proxy (NPP Proxy) 生成逻辑总结

## 概述

Net Productivity Proxy 是基于 MODIS MOD13Q1 NDVI 数据计算的年度植被生产力代理指标，用于 SDG 15.3.1 土地退化评估。该指标通过计算生长季窗口内 NDVI 的积分（AUC, Area Under Curve）来表征年度植被生产力水平。

## 核心生成流程

### 1. 峰值确定（Peak DOY 计算）

**目的**：确定研究区（ROI）内植被生长最旺盛的时间点，用于定义统一的生长季统计窗口。

**步骤**：

1. **数据准备**
   - 使用多年（如 2000-2015）的 MOD13Q1 NDVI 数据
   - 每期影像代表 16 天的合成数据（约 23 期/年）

2. **ROI 平均 NDVI 时间序列计算**
   - 对每一期影像，计算 ROI 内所有像素的 NDVI 平均值
   - 得到多年混合的 NDVI 时间序列数据

3. **时间分组（bin16）**
   - 将一年分为 23 个 bin，每个 bin 代表 16 天
   - 计算每个 bin16 的多年平均 NDVI 值

4. **峰值识别**
   - 找到多年平均 NDVI 最大的 bin16
   - 计算对应的峰值 DOY（Day of Year）：
     ```
     peak_doy = (peak_bin - 1) * 16 + 8
     ```
   - 其中 `peak_bin` 为最大平均 NDVI 对应的 bin 编号，8 为 bin 中心偏移

**输出**：`peak_doy` - 研究区植被生长峰值对应的年内日期

---

### 2. 生长季窗口定义

**目的**：基于峰值 DOY 定义统一的统计窗口，用于后续逐像素的年度 AUC 计算。

**窗口参数**：
- `W_DAYS = 64`：窗口半宽（天），约 4 个 16 天周期
- `start_doy = peak_doy - W_DAYS`
- `end_doy = peak_doy + W_DAYS`

**窗口特点**：
- 全局统一：所有像素使用相同的窗口
- 自适应：基于 ROI 平均生长季特征确定
- 稳健性：避免单像素噪声影响窗口选择

**示例**：
- 若 `peak_doy = 24`，则窗口为 `[-40, 88]`（处理跨年情况）
- 实际窗口约为 1 月下旬至 3 月底（约 128 天）

---

### 3. 年度 AUC 计算（逐像素）

**目的**：对每个像素、每一年，计算生长季窗口内的 NDVI 积分作为年度生产力代理。

**计算步骤**：

1. **数据预处理**
   ```python
   def prep_full(img):
       ndvi = img.select("NDVI").multiply(0.0001).rename("ndvi")
       doy = ee.Date(img.get("system:time_start")).getRelative("day", "year").add(1)
       doy_band = ee.Image.constant(doy).rename("doy").toInt16()
       return ndvi.addBands(doy_band)
   ```
   - NDVI 缩放：原始值 × 0.0001（MODIS 缩放因子）
   - DOY 计算：获取影像对应的年内日期

2. **窗口筛选**
   ```python
   def in_window(img):
       doy = img.select("doy")
       return img.updateMask(doy.gte(start_doy).And(doy.lte(end_doy)))
   ```
   - 仅保留窗口内的 NDVI 影像
   - 窗口外像素被掩膜（mask）

3. **年度 AUC 计算**
   ```python
   def annual_auc(y):
       year_ic = ic_full.filterDate(ys, ye).map(in_window)
       auc = year_ic.select("ndvi").sum().multiply(STEP_DAYS).rename("auc")
       return auc
   ```
   - **公式**：`AUC = Σ(NDVI_i) × STEP_DAYS`
   - 其中 `STEP_DAYS = 16`（每期代表 16 天）
   - 对窗口内所有有效 NDVI 值求和后乘以时间步长

**输出**：
- 年度 `ImageCollection`：每年一张 AUC 影像
- 空间分辨率：250 m（与 MOD13Q1 一致）
- 时间范围：2000-2015（或指定年份范围）

---

### 4. 质量控制（QA/QC）

**目的**：识别数据缺失区域和长期低生产力区域，确保后续趋势/状态分析的可靠性。

#### 4.1 有效年份计数（valid_years）

**定义**：每个像素在时间序列中非空（unmasked）的年份数

**计算**：
```python
valid_years = annual_auc_ic.select([band]).count().rename("valid_years")
```

**用途**：
- `valid_years = 0`：完全无数据区域，需剔除
- `valid_years < 8`：缺年严重，趋势估计不稳定
- `valid_years ≈ 总年数`：数据充足，可稳健分析


---

## 关键参数

| 参数 | 值 | 说明 |
|------|-----|------|
| `STEP_DAYS` | 16 | MOD13Q1 时间分辨率（天） |
| `W_DAYS` | 64 | 窗口半宽（天），约 4 个周期 |
| `SCALE` | 250 | 输出空间分辨率（米） |
| `PEAK_BASE_START_YEAR` | 2000 | 峰值计算起始年份 |
| `PEAK_BASE_END_YEAR` | 2015 | 峰值计算结束年份 |

---

## 输出产品

1. **年度 AUC 影像集合**：`annual_auc_ic`
   - 格式：`ee.ImageCollection`
   - 波段：`auc`（单波段）
   - 时间跨度：2000-2015（16 年）

2. **QA/QC 栅格**：
   - `valid_years`：有效年份计数图
   - `mean_auc`：多年平均生产力图
   - `low_prod_mask`：低生产力掩膜

3. **导出格式**：
   - 多波段堆叠：`annual_auc_ic.toBands()`（每波段 = 一年）
   - 导出至 Google Drive，后续下载至本地

---

## 方法优势

1. **稳健性**：使用 ROI 平均曲线确定窗口，避免单像素噪声
2. **可解释性**：基于生长季特征的窗口定义，符合生态学原理
3. **一致性**：全局统一窗口，便于跨年、跨像素比较
4. **适应性**：窗口随研究区特征自动调整（peak_doy 自适应）

---

## 后续应用

生成的年度 AUC 数据用于：

1. **趋势分析（Trend）**：
   - Theil-Sen slope 估计
   - Mann-Kendall 显著性检验

2. **状态评估（State）**：
   - 基准期 vs 评估期对比
   - Improved/Stable/Degraded 分类

3. **综合评估**：
   - 与土地覆被变化（LC）、土壤有机碳（SOC）结合
   - SDG 15.3.1 土地退化综合判定

### 计算逐像素的net productivity proxy


In [55]:
from __future__ import annotations
import ee


In [56]:
import ee
ee.Initialize(project='robust-builder-418813')

resp = ee.data.listOperations()

# 兼容：resp 可能是 list，也可能是 dict
if isinstance(resp, list):
    operations = resp
else:
    operations = resp.get("operations", [])

print("operations:", len(operations))

for op in operations:
    # op 可能是 dict（最常见）
    if isinstance(op, dict):
        name = op.get("name")
        state = op.get("metadata", {}).get("state")
        if state in ("RUNNING", "PENDING") and name:
            ee.data.cancelOperation(name)
            print("canceled:", name, state)


operations: 140


In [73]:
# -----------------------------
# 1) 你需要改的参数
# -----------------------------
ROI_ASSET = "projects/robust-builder-418813/assets/abu_dhabi_all"
ROI = ee.FeatureCollection(ROI_ASSET).geometry()
roi_region = ROI.bounds()

START_YEAR = 2000
END_YEAR = 2015

# 用哪些年份来找 peak（多年平均曲线）
PEAK_BASE_START_YEAR = 2000
PEAK_BASE_END_YEAR = 2015

# 窗口半宽（天），例如 64 天 ~ 4 个 16-day 期
W_DAYS = 64

# NDVI 质量控制：可选（建议至少做个简单阈值/QA）
USE_QA = False

# AUC 近似：每景代表 16 天
STEP_DAYS = 16

In [34]:
# 1) ROI 的几何类型 + bounds（最直观）
print("ROI type:", ROI.type().getInfo())
print("ROI bounds:", ROI.bounds().coordinates().getInfo())  # 看数字是不是 lon/lat 量级（50多, 20多）

# 2) ROI 的中心点（应在 Abu Dhabi 附近）
centroid = ROI.centroid()
print("ROI centroid:", centroid.coordinates().getInfo())    # 期望 [l]()


ROI type: MultiPolygon
ROI bounds: [[[51.49979455735682, 22.631512939978958], [56.38323257373369, 22.631512939978958], [56.38323257373369, 26.069490395660484], [51.49979455735682, 26.069490395660484], [51.49979455735682, 22.631512939978958]]]
ROI centroid: [54.32311808848977, 23.91713007540978]


### STEP one:生成csv:每一行为一个dictionary，里面保存每张图像的平均值

In [ ]:
STEP_DAYS = 16

def prep(img):
    ndvi = img.select("NDVI").multiply(0.0001).rename("ndvi")
    #每一张图像的时间戳属性叫system:time_start；得到 年内第几天（从 0 开始）；.add(1)把 DOY 变成从 1 开始
    doy = ee.Date(img.get("system:time_start")).getRelative("day", "year").add(1)
    #把 DOY 分到 16 天一组的编号里（bin16）
    bin16 = ee.Number(doy).subtract(1).divide(STEP_DAYS).floor().add(1)
    #你返回的是 一张只有 ndvi 波段的影像，并把一些属性挂上去
    return (ndvi
            .set("system:time_start", img.get("system:time_start"))
            .set("doy", doy)
            .set("bin16", bin16))

ic = (ee.ImageCollection("MODIS/061/MOD13Q1")
      .filterDate(f"{PEAK_BASE_START_YEAR}-01-01", f"{PEAK_BASE_END_YEAR+1}-01-01")
      .map(prep))

def img_to_row(img):
    #输出 d 是一个 ee.Dictionary
    d = img.reduceRegion(
        reducer=ee.Reducer.mean(), #对于一张图做reduce，一张图得到一个值
        geometry=ROI,
        scale=1000,          # ✅ 粗一点，够用了
        maxPixels=1e13,
        bestEffort=True,
        tileScale=4,
    )
    # 用 contains 容错，避免空字典
    #从一个字典/属性表 d 里取 "ndvi"，如果没有这个键，就用 -9999 作为默认值，然后再转成 ee.Number。
    #ee.Number 是 Google Earth Engine 的“服务端数值对象”（server-side object），代表一个标量数字（类似 3、0.12 这种
    ndvi_mean = ee.Number(ee.Algorithms.If(d.contains("ndvi"),
                                        d.get("ndvi"),  #如果字典d里有ndvi键，就取它的值
                                        -9999))         #否则就用-9999作为默认值
    #创建一个没有几何形状的 feature（几何是 None）
    return ee.Feature(None, {
        "time": img.get("system:time_start"),
        "doy": img.get("doy"),
        "bin16": img.get("bin16"),
        "roi_mean_ndvi": ndvi_mean
    })

ts_fc = ee.FeatureCollection(ic.map(img_to_row)).filter(ee.Filter.gt("roi_mean_ndvi", -9990))

task = ee.batch.Export.table.toDrive(
    collection=ts_fc,
    description="ROI_NDVI_mean_timeseries",
    fileFormat="CSV"
)
task.start()
print("Export started: ROI_NDVI_mean_timeseries")

Export started: ROI_NDVI_mean_timeseries


In [ ]:
peak_doy = 2
W_DAYS = 64
STEP_DAYS = 16
#把 Python 里的 peak_doy（普通数字）转成 GEE 的 服务端数值 ee.Number
peak_doy_ee = ee.Number(peak_doy)
start_doy = peak_doy_ee.subtract(W_DAYS)
end_doy = peak_doy_ee.add(W_DAYS)

def prep_full(img):
    ndvi = img.select("NDVI").multiply(0.0001).rename("ndvi")
    p = ndvi.projection()
    doy = ee.Date(img.get("system:time_start")).getRelative("day", "year").add(1)
    #把这个 DOY（一个数）做成一张“常数影像”（每个像素值都等于这个 DOY，命名为波段 "doy"
    doy_band = ee.Image.constant(doy).rename("doy").toInt16()
    #如果集合里影像的默认投影不一致或某些操作让投影变得“未定义/混杂”，GEE 可能用一个非常粗糙的默认投影
    #后续sum会出错，所以必须把新增的band的的坐标系确定为一致的
    ndvi = ndvi.reproject(p)
    #把原影像的 "system:time_start" 属性复制过来（保持时间信息）
    return ndvi.addBands(doy_band).copyProperties(img, ["system:time_start"])

ic_full = (ee.ImageCollection("MODIS/061/MOD13Q1")
           .filterDate(f"{START_YEAR}-01-01", f"{END_YEAR+1}-01-01")
           .map(prep_full))
#输入一张影像（已经有 doy 波段）
#in_window 是在根据影像日期，把整张影像“保留”或“屏蔽掉”
def in_window(img):
    doy = img.select("doy").toInt16()
    #这行 判断窗口是否为正常不跨年的情况，start_doy >= 1 and end_doy <= 366 and start_doy <= end_doy
    normal = ee.Number(start_doy).gte(1).And(ee.Number(end_doy).lte(366)).And(ee.Number(start_doy).lte(end_doy))
    #正常窗口的掩膜：DOY 在 [start, end] 内就保留
    def normal_mask():
        return doy.gte(start_doy).And(doy.lte(end_doy))
   
    def wrap_mask():
        return doy.gte(ee.Number(start_doy).add(366)).Or(doy.lte(end_doy))
    #不在窗口内的会被mask掉，因为doy都是一样的数，所以整张图像都会mask/保留
    mask = ee.Image(ee.Algorithms.If(normal, normal_mask(), wrap_mask()))
    return img.updateMask(mask)

ref_proj = ee.ImageCollection("MODIS/061/MOD13Q1").first().select("NDVI").projection()

def annual_auc(y):
    y = ee.Number(y)
    ys = ee.Date.fromYMD(y, 1, 1) #start year
    ye = ys.advance(1, "year") #end year,往后推一年
    #只选取当年 y的图像，通过window函数过滤，只保留在窗口内的推向
    year_ic = ic_full.filterDate(ys, ye).map(in_window)
   #创建一个全为空的影像 = 0 
    empty_auc = (
    ee.Image.constant(0).toFloat()
    .rename("auc")
    .updateMask(ee.Image.constant(0))   # 这里作为 mask，类型无所谓
)
    
    year_auc = (
        year_ic.select("ndvi")
        .sum()
        .multiply(ee.Number(STEP_DAYS)) #离散微积分，乘以16得到面积
        .rename("auc")
        .toFloat()
        .setDefaultProjection(ref_proj)
    )

    auc = ee.Image(
        ee.Algorithms.If(year_ic.size().gt(0), year_auc, empty_auc)
    )


    return (auc.clip(ROI)
              .set("year", y)
              .set("system:time_start", ys.millis()))

#2000-2015
years = ee.List.sequence(START_YEAR, END_YEAR)
annual_auc_ic = ee.ImageCollection(years.map(annual_auc))

# 这句通常就不会再触发“并发聚合爆炸”了
print("annual auc n_images:", annual_auc_ic.size().getInfo())


annual auc n_images: 16


In [79]:
#这个单元格只用来检验是不是ndvi一开始就是偏的
import math
dx = 0.0025
xmin = 51.49979455735682
xmax = 56.38323257373369
ymin = 22.631512939978958
ymax = 26.069490395660484

width  = math.ceil((xmax - xmin) / dx)
height = math.ceil((ymax - ymin) / dx)

crsTransform = [dx, 0, xmin, 0, -dx, ymax]  # 4326: 左上角原点 + 像素大小
def prep_evi(img):
    evi = img.select("EVI").multiply(0.0001).rename("evi")

    # 用 evi 自己的原生投影锁住（SR-ORG:6974 + 231.656m transform）

    doy = ee.Date(img.get("system:time_start")).getRelative("day", "year").add(1)
    doy_band = ee.Image.constant(doy).rename("doy").toInt16()
    return evi.addBands(doy_band).copyProperties(img, ["system:time_start"])


ic_full_ = (ee.ImageCollection("MODIS/061/MOD13Q1")
            .filterDate(f"{START_YEAR}-01-01", f"{END_YEAR+1}-01-01")
            .map(prep_evi))
evi = ic_full_.select("evi")
p_evi = ic_full_.first().select("evi").projection()
year_auc = (ic_full_.select("evi")
            .sum()
            .multiply(STEP_DAYS)
            .rename("auc")
            .toFloat())

year_auc_4326 = year_auc.clipToBoundsAndScale(
    geometry=ROI,
    scale=0.0025   # 度单位
).reproject("EPSG:4326", None, 0.0025)
        
task = ee.batch.Export.image.toDrive(
    image=year_auc_4326,
    description=f"evi_auc_{START_YEAR}_{END_YEAR}_4326",
    folder="GEE",
    fileNamePrefix=f"evi_auc_{START_YEAR}_{END_YEAR}_4326",
    region=roi_region,
    crs="EPSG:4326",
    scale=0.0025,
    maxPixels=1e13,
)
task.start()


In [75]:
print("ROI bounds:", ee.FeatureCollection(ROI).geometry().bounds().getInfo())



ROI bounds: {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[51.49979455735682, 22.631512939978958], [56.38323257373369, 22.631512939978958], [56.38323257373369, 26.069490395660484], [51.49979455735682, 26.069490395660484], [51.49979455735682, 22.631512939978958]]]}


In [ ]:
# annual_auc_ic: ee.ImageCollection
stacked = annual_auc_ic.toBands()   # ee.Image，多波段：每个band=一年
#通过连接gee,导出到本地
task = ee.batch.Export.image.toDrive(
    image=stacked,
    description="annual_auc_toBands_2000_2015",
    folder="GEE",
    fileNamePrefix="annual_auc_toBands_2000_2015",
    region=ROI,          # ee.Geometry / GeoJSON
    scale=250,           # 你的数据分辨率
    maxPixels=1e13
)
task.start()4
print("Export started:", task.id)

Export started: QIBLX6CPZR4XWBBIY3MLXF3J


In [35]:
# 生成多年的
def generate_ndvi(reporting_year:int):
  PEAK_BASE_START_YEAR = reporting_year - 15
  PEAK_BASE_END_YEAR = reporting_year
  ic = (ee.ImageCollection("MODIS/061/MOD13Q1")
    .filterDate(f"{PEAK_BASE_START_YEAR}-01-01", f"{PEAK_BASE_END_YEAR+1}-01-01")
    .map(prep))
  ts_fc = ee.FeatureCollection(ic.map(img_to_row)).filter(ee.Filter.gt("roi_mean_ndvi", -9990))

  task = ee.batch.Export.table.toDrive(
      collection=ts_fc,
      description=f"ROI_NDVI_mean_timeseries_{reporting_year}",
      fileFormat="CSV"
  )
  task.start()
  print(f"Export started {reporting_year}: ROI_NDVI_mean_timeseries")
for year in range(2015,2026):
  generate_ndvi(year)


Export started 2015: ROI_NDVI_mean_timeseries
Export started 2016: ROI_NDVI_mean_timeseries
Export started 2017: ROI_NDVI_mean_timeseries
Export started 2018: ROI_NDVI_mean_timeseries
Export started 2019: ROI_NDVI_mean_timeseries
Export started 2020: ROI_NDVI_mean_timeseries
Export started 2021: ROI_NDVI_mean_timeseries
Export started 2022: ROI_NDVI_mean_timeseries
Export started 2023: ROI_NDVI_mean_timeseries
Export started 2024: ROI_NDVI_mean_timeseries
Export started 2025: ROI_NDVI_mean_timeseries


In [30]:
ref_proj = ee.ImageCollection("MODIS/061/MOD13Q1").first().select("NDVI").projection()
ref_info = ref_proj.getInfo()

crs = ref_info["crs"]
t = ref_info["transform"]   # ✅ List[float] 长度 6

In [ ]:
import pandas as pd
def year_count(y):
    y = ee.Number(y)
    ys = ee.Date.fromYMD(y, 1, 1)
    ye = ys.advance(1, "year")
    n = ic_full.filterDate(ys, ye).size()
    return ee.Feature(None, {"year": y, "n": n})

def generate_npp(reporting_year: int):
    ROI_ASSET = "projects/robust-builder-418813/assets/abu_dhabi_all"
    ROI = ee.FeatureCollection(ROI_ASSET).geometry()
    df = pd.read_csv(f"../data/NDVI_MEAN/ROI_NDVI_mean_timeseries_{reporting_year}.csv")
    START_YEAR = reporting_year - 15
    END_YEAR = reporting_year
    print(START_YEAR, END_YEAR)
   #计算peak day of year
    g = df.groupby("bin16")["roi_mean_ndvi"].mean()
    peak_bin = int(g.idxmax())
    peak_doy = (peak_bin - 1) * 16 + 8
    print("peak_bin:", peak_bin, "peak_doy:", peak_doy)
  
    W_DAYS = 64
    STEP_DAYS = 16
    peak_doy_ee = ee.Number(peak_doy)
    start_doy = peak_doy_ee.subtract(W_DAYS)
    end_doy = peak_doy_ee.add(W_DAYS)

    # ---- 绑定本次参数的 year_count / annual_auc ----
    def year_count_local(y):
        y = ee.Number(y)
        ys = ee.Date.fromYMD(y, 1, 1)
        ye = ys.advance(1, "year")
        n = ic_full.filterDate(ys, ye).size()
        return ee.Feature(None, {"year": y, "n": n})
    def prep_full(img):
        ndvi = img.select("NDVI").multiply(0.0001).rename("ndvi")
        doy = ee.Date(img.get("system:time_start")).getRelative("day", "year").add(1)
        #把这个 DOY（一个数）做成一张“常数影像”（每个像素值都等于这个 DOY，命名为波段 "doy"
        doy_band = ee.Image.constant(doy).rename("doy").toInt16()
        #把原影像的 "system:time_start" 属性复制过来（保持时间信息）
        return ndvi.addBands(doy_band).copyProperties(img, ["system:time_start"])

    def in_window(img):
        doy = img.select("doy").toInt16()
        #这行 判断窗口是否为正常不跨年的情况，start_doy >= 1 and end_doy <= 366 and start_doy <= end_doy
        normal = ee.Number(start_doy).gte(1).And(ee.Number(end_doy).lte(366)).And(ee.Number(start_doy).lte(end_doy))
        #正常窗口的掩膜：DOY 在 [start, end] 内就保留
        def normal_mask():
            return doy.gte(start_doy).And(doy.lte(end_doy))
    
        def wrap_mask():
            return doy.gte(ee.Number(start_doy).add(366)).Or(doy.lte(end_doy))
        #不在窗口内的会被mask掉，因为doy都是一样的数，所以整张图像都会mask/保留
        mask = ee.Image(ee.Algorithms.If(normal, normal_mask(), wrap_mask()))
        return img.updateMask(mask)

    def annual_auc_local(y):
        y = ee.Number(y)
        ys = ee.Date.fromYMD(y, 1, 1)
        ye = ys.advance(1, "year")
        #year_ic只取当年的影像
        year_ic = ic_full.filterDate(ys, ye).map(in_window)

        empty_auc = ee.Image.constant(0).rename("auc").updateMask(ee.Image.constant(0))

        auc = ee.Image(
            ee.Algorithms.If(
                year_ic.size().gt(0),
                year_ic.select("ndvi").sum().multiply(STEP_DAYS).rename("auc"),
                empty_auc
            )
        ).setDefaultProjection(ref_proj)

        return (auc.clip(ROI)
                .set("year", y)
                .set("system:time_start", ys.millis()))
    #--------------------------------
    #ic_full 是所有年份的影像
    ic_full = (ee.ImageCollection("MODIS/061/MOD13Q1")
               .filterDate(f"{START_YEAR}-01-01", f"{END_YEAR+1}-01-01")
               .map(prep_full))

    # sanity check
    print("ic_full first date =", ee.Date(ic_full.first().get("system:time_start")).format("YYYY-MM-dd").getInfo())
    print("ic_full last date  =", ee.Date(ic_full.sort("system:time_start", False).first().get("system:time_start")).format("YYYY-MM-dd").getInfo())

    ref_proj = ee.ImageCollection("MODIS/061/MOD13Q1").first().select("NDVI").projection()
    years = ee.List.sequence(START_YEAR, END_YEAR)

    # counts
    counts = ee.FeatureCollection(years.map(year_count_local))
    print(counts.aggregate_array("year").getInfo())
    print(counts.aggregate_array("n").getInfo())

    annual_auc_ic = ee.ImageCollection(years.map(annual_auc_local))
    print("annual auc n_images:", annual_auc_ic.size().getInfo())

    stacked = annual_auc_ic.toBands().clip(ROI)
    print("ROI bounds:", ROI.bounds().getInfo())
    print("stacked projection:", stacked.projection().getInfo())


    task = ee.batch.Export.image.toDrive(
        image=stacked,
        description=f"annual_auc_toBands_{START_YEAR}_{END_YEAR}",
        folder="NDVI_PROXY",
        fileNamePrefix=f"annual_auc_toBands_{START_YEAR}_{END_YEAR}",
        region=ROI,
        crs="EPSG:4326",
        scale= 250,
        maxPixels=1e13,
    )
    task.start()
    print("Export started:", task.id)

    
    

In [84]:
for year in range(2015,2026):
  generate_npp(year)

2000 2015
peak_bin: 2 peak_doy: 24
ic_full first date = 2000-02-18
ic_full last date  = 2015-12-19
[2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015]
[20, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23]
annual auc n_images: 16
Export started: FCJSCGOENSL6GYPDMHTKRT65
2001 2016
peak_bin: 2 peak_doy: 24
ic_full first date = 2001-01-01
ic_full last date  = 2016-12-18
[2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016]
[23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23]
annual auc n_images: 16
Export started: EG5ZGY2OZCRW7SU6YKWAQ3XH
2002 2017
peak_bin: 2 peak_doy: 24
ic_full first date = 2002-01-01
ic_full last date  = 2017-12-19
[2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017]
[23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23]
annual auc n_images: 16
Export started: KZTXAYOYI34MOUQHRIEWRQF5
2003 2018
peak_bin: 2 

In [29]:
info = ref_proj.getInfo()
print(info.keys())
print(info)


dict_keys(['type', 'crs', 'transform'])
{'type': 'Projection', 'crs': 'SR-ORG:6974', 'transform': [231.65635826395825, 0, -20015109.354, 0, -231.65635826395834, 10007554.677003]}
